In [ ]:
import pandas as pd
import openai

from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

In [ ]:
data = pd.read_json("../../data/meta_Books_2022_2023_with_category_ratings_10_sample_1000.jsonl", lines=True)

In [ ]:
data.head()

In [ ]:
data['features'][0]

In [ ]:
data['images'][0]

### Preprocess Title and Features

In [ ]:
def process_description(row):
    return f"{row['title']} {' '.join(row['features'])}"

def extract_first_large_image(row):
    return row['images'][0].get('large', '') if row['images'] else ''

In [ ]:
data['description'] = data.apply(process_description, axis=1)
data['image'] = data.apply(extract_first_large_image, axis=1)

## Sample 50 items form teh dataset

In [ ]:
df_sample = data.sample(50, random_state=42)

In [ ]:
df_to_embed = df_sample[
    ["title", "description", "categories_middle", "image", "rating_number", "average_rating", "price", "parent_asin"]
].to_dict(orient="records")

In [ ]:
df_to_embed

### Define the embedding function

In [ ]:
response = openai.embeddings.create(
    input="Random text", model="text-embedding-3-small"
)

In [ ]:
len(response.data[0].embedding)

In [ ]:
def get_embedding(text, model='text-embedding-3-small'):

    response = openai.embeddings.create(
        input=text,
        model=model
    )

    return response.data[0].embedding

In [ ]:
qdrant_client = QdrantClient(url="http://localhost:6333")

In [ ]:
qdrant_client.create_collection(
    collection_name="Amazon-items-collection-01",
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE)
)

### Embed Data

#### Test

In [ ]:
pointstruct = PointStruct(
    id=0,
    vector=get_embedding("Test text"),
    payload={
        "text": "Test text",
        "model": "text-embedding-3-small"
    }
)

In [ ]:
pointstruct

#### Amazon data

In [ ]:
df_to_embed

In [ ]:
pointstructs = []
for idx, data in enumerate(df_to_embed):
    embedding = get_embedding(data['description'])
    pointstructs.append(
        PointStruct(
            id=idx,
            vector=embedding,
            payload=data
        )
    )


In [ ]:
pointstructs

### Write embedded data to Qdrant

In [ ]:
qdrant_client.upsert(
    collection_name="Amazon-items-collection-01",
    points=pointstructs
)

### Define function for data retrieval

In [ ]:
def retrieve_data(query, k=5):
    query_embedding = get_embedding(query)
    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01",
        query=query_embedding,
        limit=k
    )
    return results

In [ ]:
retrieve_data("Cooking recepies").points